# Deep Learning — Advanced Reference
> **Level:** Advanced | **Goal:** Build, train, and debug neural networks with PyTorch

## Table of Contents
1. [PyTorch Core Concepts](#core)
2. [Feedforward Networks (MLP)](#mlp)
3. [Training Loop & Best Practices](#training)
4. [Convolutional Neural Networks (CNN)](#cnn)
5. [Recurrent Networks (LSTM/GRU)](#rnn)
6. [Transfer Learning](#transfer)
7. [Regularization Techniques](#regularization)
8. [Learning Rate Scheduling](#lr)
9. [Transformers & Attention](#transformers)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms, models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
torch.manual_seed(42)

---
## 1 · PyTorch Core Concepts <a id='core'></a>

In [ ]:
# ── Tensors ────────────────────────────────────────────────────
x = torch.tensor([[1., 2.], [3., 4.]], requires_grad=True)

# Operations on tensors preserve the computation graph
y = x ** 2 + 2 * x + 1
loss = y.mean()
loss.backward()          # compute gradients
print("Gradient of loss w.r.t. x:")
print(x.grad)            # dy/dx = 2x + 2

In [ ]:
# ── Useful tensor operations ───────────────────────────────────
a = torch.randn(3, 4)

print("Shape:",     a.shape)
print("dtype:",     a.dtype)
print("Sum dim=1:", a.sum(dim=1))     # (3,)
print("Unsqueeze:", a.unsqueeze(0).shape)   # (1,3,4)
print("Flatten:",   a.view(-1).shape)       # (12,)

# Mask (boolean indexing)
mask = a > 0
print("Positive values:", a[mask].shape)

# Broadcasting
b = torch.randn(4)
print("a + b shape:", (a + b).shape)   # (3,4) + (4,) → (3,4)

---
## 2 · Feedforward Network (MLP) <a id='mlp'></a>

In [ ]:
# ── Dataset: tabular classification ───────────────────────────
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=5000, n_features=20, n_informative=12,
                            n_redundant=4, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

# Convert to tensors
X_tr_t = torch.FloatTensor(X_tr);  y_tr_t = torch.LongTensor(y_tr)
X_te_t = torch.FloatTensor(X_te);  y_te_t = torch.LongTensor(y_te)

train_ds = TensorDataset(X_tr_t, y_tr_t)
val_size = int(0.15 * len(train_ds))
train_ds, val_ds = random_split(train_ds, [len(train_ds)-val_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256)

In [ ]:
# ── MLP with BatchNorm and Dropout ─────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout_rate=0.3):
        super().__init__()
        layers = []
        dims = [input_dim] + hidden_dims

        for i in range(len(hidden_dims)):
            layers += [
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]

        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        self.net = nn.Sequential(*layers)

        # Weight initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

model = MLP(input_dim=20, hidden_dims=[256, 128, 64], output_dim=2, dropout_rate=0.3)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")
print(model)

---
## 3 · Training Loop & Best Practices <a id='training'></a>

In [ ]:
# ── Professional training loop with early stopping ─────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, n_correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        n_correct  += (logits.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, n_correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, n_correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = model(X_batch)
        total_loss += criterion(logits, y_batch).item() * len(y_batch)
        n_correct  += (logits.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, n_correct / n

# ── Training configuration ─────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-2, epochs=40, steps_per_epoch=len(train_loader)
)

# ── Train with early stopping ─────────────────────────────────
best_val_loss = float('inf')
patience, patience_counter = 8, 0
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(40):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d} | train: loss={tr_loss:.4f} acc={tr_acc:.3f} | "
              f"val: loss={val_loss:.4f} acc={val_acc:.3f}")

# Load best model
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
_, test_acc = evaluate(model, test_loader, criterion)
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val',   color='crimson')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy Loss')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train', color='steelblue')
axes[1].plot(history['val_acc'],   label='Val',   color='crimson')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend()

[ax.spines[['top','right']].set_visible(False) for ax in axes]
plt.suptitle('Training History', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4 · Convolutional Neural Networks (CNN) <a id='cnn'></a>

In [ ]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU → optional MaxPool."""
    def __init__(self, in_ch, out_ch, kernel=3, pool=False):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CNN(nn.Module):
    """Small CNN for 28×28 grayscale images (MNIST-like)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,  32, pool=True),    # 28→14
            ConvBlock(32, 64, pool=True),    # 14→7
            ConvBlock(64, 128),              # 7→7
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # global average pool → (B,128,1,1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


cnn = CNN(num_classes=10)
dummy_input = torch.randn(4, 1, 28, 28)   # batch=4, channels=1, 28×28
output = cnn(dummy_input)
print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print(f"Params: {sum(p.numel() for p in cnn.parameters()):,}")

---
## 5 · Recurrent Networks: LSTM/GRU <a id='rnn'></a>

In [ ]:
class LSTMPredictor(nn.Module):
    """
    Many-to-one LSTM for sequence classification.
    Input: (batch, seq_len, input_size)
    Output: (batch, num_classes)
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True     # bi-directional doubles hidden size
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Take output from last time step of both directions
        # h_n shape: (num_layers*2, batch, hidden_size)
        last_fwd = h_n[-2, :, :]   # last layer, forward
        last_bwd = h_n[-1, :, :]   # last layer, backward
        combined = torch.cat([last_fwd, last_bwd], dim=1)

        out = self.dropout(combined)
        return self.fc(out)


lstm_model = LSTMPredictor(input_size=10, hidden_size=64, num_layers=2,
                            num_classes=3, dropout=0.3)
seq_input = torch.randn(8, 20, 10)  # batch=8, seq_len=20, features=10
print("LSTM output shape:", lstm_model(seq_input).shape)

---
## 6 · Transfer Learning <a id='transfer'></a>

In [ ]:
# ── Pattern: fine-tune a pretrained backbone ───────────────────
def build_transfer_model(backbone_name: str, num_classes: int, freeze_backbone: bool = True):
    """
    Load a pretrained backbone and replace the classifier head.

    Strategy:
    - freeze_backbone=True  → train only the new head (feature extraction)
    - freeze_backbone=False → fine-tune entire network (unfreeze after head converges)
    """
    # Load pretrained model
    backbone = getattr(models, backbone_name)(weights='DEFAULT')

    if freeze_backbone:
        for param in backbone.parameters():
            param.requires_grad = False

    # Replace the final FC layer
    if hasattr(backbone, 'fc'):
        in_features = backbone.fc.in_features
        backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    elif hasattr(backbone, 'classifier'):
        in_features = backbone.classifier[-1].in_features
        backbone.classifier[-1] = nn.Linear(in_features, num_classes)

    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    total = sum(p.numel() for p in backbone.parameters())
    print(f"{backbone_name}: {trainable:,} / {total:,} params trainable ({100*trainable/total:.1f}%)")
    return backbone

model_tl = build_transfer_model('resnet18', num_classes=5, freeze_backbone=True)

# Two-phase fine-tuning:
# Phase 1: train only head with high LR
print("\nPhase 1: head only")
optimizer_tl = optim.AdamW(
    filter(lambda p: p.requires_grad, model_tl.parameters()),
    lr=1e-3
)

# Phase 2: unfreeze and fine-tune with low LR + discriminative LRs
print("\nPhase 2: unfreeze all (discriminative LR)")
for param in model_tl.parameters():
    param.requires_grad = True

optimizer_tl = optim.AdamW([
    {'params': model_tl.layer1.parameters(), 'lr': 1e-5},
    {'params': model_tl.layer2.parameters(), 'lr': 2e-5},
    {'params': model_tl.layer3.parameters(), 'lr': 5e-5},
    {'params': model_tl.layer4.parameters(), 'lr': 1e-4},
    {'params': model_tl.fc.parameters(),     'lr': 1e-3},
])
print("Discriminative learning rates set.")

---
## 7 · Regularization Techniques <a id='regularization'></a>

| Technique | How | When |
|---|---|---|
| Dropout | Randomly zero activations | Dense layers, after attention |
| BatchNorm | Normalize layer activations | After conv/linear, before activation |
| LayerNorm | Normalize per sample | Transformers, RNNs |
| Weight decay (L2) | Penalize large weights | Optimizer: `weight_decay=1e-4` |
| Gradient clipping | Cap gradient norm | Prevents exploding gradients in RNNs |
| Label smoothing | Soft labels | Classification with overconfident models |
| Data augmentation | Random transforms | Vision tasks |

In [ ]:
# ── Label smoothing loss ───────────────────────────────────────
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, eps: float = 0.1):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_probs = F.log_softmax(pred, dim=-1)
        # Standard CE part
        nll_loss = -log_probs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        # Smoothed part
        smooth_loss = -log_probs.mean(dim=-1)
        return ((1 - self.eps) * nll_loss + self.eps * smooth_loss).mean()

# Test
logits = torch.randn(4, 10)
labels = torch.randint(0, 10, (4,))
ce = nn.CrossEntropyLoss()(logits, labels)
ls = LabelSmoothingCrossEntropy(eps=0.1)(logits, labels)
print(f"CE: {ce:.4f} | Label Smoothing: {ls:.4f}")

# ── Data augmentation (torchvision) ───────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomCrop(224, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("Augmentation pipelines defined.")

---
## 8 · Learning Rate Scheduling <a id='lr'></a>

| Scheduler | When to use |
|---|---|
| `OneCycleLR` | Default choice — warm-up + annealing |
| `CosineAnnealingLR` | Long runs, restart tricks |
| `ReduceLROnPlateau` | When validation stops improving |
| `ExponentialLR` | Simple exponential decay |
| Linear warmup + cosine | Transformers |

In [ ]:
# ── Visualize LR schedules ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
n_steps = 200

dummy_model = nn.Linear(10, 1)
schedules = [
    ('OneCycleLR', optim.SGD(dummy_model.parameters(), lr=0.01),
     lambda opt: optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=n_steps)),
    ('CosineAnnealing', optim.SGD(dummy_model.parameters(), lr=0.1),
     lambda opt: optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=50, T_mult=2)),
    ('ReduceOnPlateau', optim.Adam(dummy_model.parameters(), lr=0.1),
     lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)),
]

for ax, (name, opt, sched_fn) in zip(axes, schedules):
    sched = sched_fn(opt)
    lrs = []
    for step in range(n_steps):
        lrs.append(opt.param_groups[0]['lr'])
        if isinstance(sched, optim.lr_scheduler.ReduceLROnPlateau):
            sched.step(1.0 - step * 0.001)  # simulate improving metric
        else:
            sched.step()
    ax.plot(lrs, color='steelblue')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Step'); ax.set_ylabel('LR')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Learning Rate Schedules', fontsize=13)
plt.tight_layout()
plt.show()

---
## 9 · Transformers & Attention <a id='transformers'></a>

In [ ]:
# ── Scaled Dot-Product Attention (from scratch) ───────────────
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.scale = d_model ** -0.5
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        # Q,K,V: (batch, heads, seq_len, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (B,H,L,L)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = self.dropout(F.softmax(scores, dim=-1))
        return torch.matmul(attn_weights, V), attn_weights  # context, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(self.d_k, dropout)

    def split_heads(self, x, B, L):
        return x.view(B, L, self.n_heads, self.d_k).transpose(1, 2)  # (B,H,L,d_k)

    def forward(self, Q, K, V, mask=None):
        B, L, _ = Q.shape
        Q = self.split_heads(self.W_q(Q), B, L)
        K = self.split_heads(self.W_k(K), B, K.shape[1])
        V = self.split_heads(self.W_v(V), B, V.shape[1])
        context, weights = self.attention(Q, K, V, mask)
        context = context.transpose(1, 2).contiguous().view(B, L, -1)
        return self.W_o(context), weights


# Test
mha = MultiHeadAttention(d_model=128, n_heads=8)
x = torch.randn(4, 16, 128)  # batch=4, seq=16, d_model=128
out, attn = mha(x, x, x)    # self-attention
print(f"MHA output: {out.shape}, attn weights: {attn.shape}")

In [ ]:
# ── Use HuggingFace Transformers for NLP tasks ─────────────────
print("""HuggingFace Transformers quick reference:

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load pretrained model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

# 2. Tokenize
inputs = tokenizer(texts, padding=True, truncation=True,
                   max_length=128, return_tensors='pt')

# 3. Fine-tune with Trainer API
args = TrainingArguments(
    output_dir='./output',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    evaluation_strategy='epoch',
    load_best_model_at_end=True,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()
""")